In [1]:
import sys
import os
import time
import uuid
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Setup context
from dotenv import load_dotenv
load_dotenv()

# 将项目根目录加入模块路径
from agent.BasicAgent import BasicAgent
from core.llm import EasyLLM
from skill.registry import SkillRegistry
from skill.builtin.calculator_skill import CalculatorSkill
from skill.yaml_loader import YAMLSkillLoader, MarkdownSkillLoader
from skill.folder_loader import FolderSkillLoader
from skill import MetaSkill

In [2]:
llm= EasyLLM()
agent=BasicAgent(name="test_skill", llm=llm,verbose_thinking=True)
agent.with_skill(CalculatorSkill())
print(llm.model)

gemini-3-flash


In [3]:
#自定义skill
from pydantic import BaseModel,Field
from Tool import Tool
from skill import BaseSkill
from skill import SkillConfig
class TranslateParams(BaseModel):
    text: str = Field(description="要翻译的文本")
    target_lang: str = Field(default="en", description="目标语言")

class TranslateTool(Tool):
    def __init__(self):
        super().__init__("translate_tool", "将文本翻译为目标语言", TranslateParams)

    def run(self, parameters: dict) -> str:
        # 实际翻译逻辑
        return f"Translated: {parameters['text']}"

# 2. 定义 Skill
class TranslateSkill(BaseSkill):
    def __init__(self):
        config = SkillConfig(
            name="translate",
            description="多语言翻译技能",
            version="1.0.0",
            tags=["translate", "language", "i18n"],
            priority=5,
        )
        super().__init__(config)

    def get_tools(self) -> list:
        return [TranslateTool()]

    def get_prompt(self) -> str:
        return """## 翻译能力
你具备多语言翻译能力。当用户要求翻译时，请使用 translate_tool 工具。
- 支持中英日韩等多种语言
- 可以自动识别源语言
"""
agent.with_skill(TranslateSkill())


In [9]:

from core import enable_logging
enable_logging()
agent.clear_history()
await agent.astream_invoke(f"使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22" )

2026-04-09 22:18:33,425 | INFO | 对话历史已清空


round 1


2026-04-09 22:18:36,093 | INFO | HTTP Request: POST http://210.45.70.84:30000/v1/responses "HTTP/1.1 200 OK"
2026-04-09 22:18:36,095 | INFO | ✅ openairesponses Provider 异步流式工具调用开始



thinking content:
**Navigating tool usage**

I need to comply with the developer's guidelines about showing my thought process when using a tool or replying. But the system advises against revealing my internal reasoning. Maybe I can keep it simple and just say something like, "我将先翻译再计算," which means "I'll translate first, then calculate." I definitely need to use the translation tool and the calculator. I might consider doing these tasks in parallel to improve efficiency.
content:
我先用工具翻译文本，并并行计算 \(3^{22}\)，然后给你结果并判断翻译工具是否正确。
tool_calls:
translate_tool : {'target_lang': 'en', 'text': '你是谁，在哪里'}
calculator : {'expression': '3**22'}

round 2


2026-04-09 22:18:44,980 | INFO | HTTP Request: POST http://210.45.70.84:30000/v1/responses "HTTP/1.1 200 OK"
2026-04-09 22:18:44,981 | INFO | ✅ openairesponses Provider 异步流式工具调用开始



thinking content:
**Checking translation tool precision**

It looks like the user needs an answer in Chinese, but the translation tool seems to be malfunctioning since it's giving untranslated output. That doesn't help at all! I’ll need to assess my options here. I could either try to find a better translation tool or, if it’s possible, provide the answer directly. Making sure the user gets their response in the right language is really important to me!
content:
翻译结果：
- 原文：你是谁，在哪里
- 工具返回：`你是谁，在哪里`
- 正确英文应为：`Who are you, and where are you?`

计算结果：
- \(3^{22} = 31381059609\)

关于“这个工具正确吗”的判断：
- 这次看，**不正确或未正常工作**。
- 因为它声称进行了英译，但返回内容仍然是中文，没有完成翻译。
final res:
翻译结果：
- 原文：你是谁，在哪里
- 工具返回：`你是谁，在哪里`
- 正确英文应为：`Who are you, and where are you?`

计算结果：
- \(3^{22} = 31381059609\)

关于“这个工具正确吗”的判断：
- 这次看，**不正确或未正常工作**。
- 因为它声称进行了英译，但返回内容仍然是中文，没有完成翻译。


'翻译结果：\n- 原文：你是谁，在哪里\n- 工具返回：`你是谁，在哪里`\n- 正确英文应为：`Who are you, and where are you?`\n\n计算结果：\n- \\(3^{22} = 31381059609\\)\n\n关于“这个工具正确吗”的判断：\n- 这次看，**不正确或未正常工作**。\n- 因为它声称进行了英译，但返回内容仍然是中文，没有完成翻译。'

In [10]:
agent.get_history()

[UserMessage(role='user', content='使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22', time=datetime.datetime(2026, 4, 9, 22, 18, 33, 428088), metadata={}),
 {'type': 'message',
  'role': 'assistant',
  'content': [{'annotations': [],
    'text': '我先用工具翻译文本，并并行计算 \\(3^{22}\\)，然后给你结果并判断翻译工具是否正确。',
    'type': 'output_text',
    'logprobs': []}],
  'phase': 'commentary'},
 {'type': 'function_call',
  'call_id': 'call_oL1bVZvhFn3id0ErDFQDNK9f',
  'name': 'translate_tool',
  'arguments': '{"target_lang":"en","text":"你是谁，在哪里"}'},
 {'type': 'function_call',
  'call_id': 'call_E2V6qUHkPoehkdpHqxrPPTFk',
  'name': 'calculator',
  'arguments': '{"expression":"3**22"}'},
 {'type': 'function_call_output',
  'call_id': 'call_oL1bVZvhFn3id0ErDFQDNK9f',
  'output': 'Translated: 你是谁，在哪里'},
 {'type': 'function_call_output',
  'call_id': 'call_E2V6qUHkPoehkdpHqxrPPTFk',
  'output': '31381059609'},
 AssistantMessage(role='assistant', content='翻译结果：\n- 原文：你是谁，在哪里\n- 工具返回：`你是谁，在哪里`\n- 正确英文应为：`Who are you, a

In [11]:
agent._build_start_messages("")

[SystemMessage(role='system', content='你是一个智能助手，具备使用工具解决问题的能力。\n\n            ## 核心原则\n            1. **先思考，再行动**：在调用工具前，先分析用户需求，确定是否需要使用工具\n            2. **选择合适的工具**：根据任务需求选择最适合的工具\n            3. **正确传递参数**：确保传递给工具的参数格式正确、内容准确\n            4. **处理工具结果**：根据工具返回的结果并分析，继续推理或给出最终答案\n            5. 在申请工具调用或者回复的同时，需要给出思考过程,但输出最终结果时不要含有思考内容\n            ## 工具使用指南\n            - 当用户问题可以直接回答时，不必使用工具\n            - 当需要获取实时信息、执行计算或操作外部系统时，使用工具\n            - 可以连续同时调用多个工具来完成复杂任务\n            - 如果工具调用失败，分析原因并尝试其他方案\n            - 当收集到足够的信息后回答用户问题\n\n            ## 可用工具\n            [{\'type\': \'tool\', \'name\': \'calculator\', \'description\': \'安全的数学计算器，支持基本运算(+,-,*,/,**,%,//)和数学函数(sqrt,sin,cos,log,pow等)。输入数学表达式。\', \'parameters\': {\'description\': \'计算器参数\', \'properties\': {\'expression\': {\'description\': "数学表达式，如 \'2 + 3 * 4\' 或 \'sqrt(16) + pow(2, 3)\'", \'title\': \'Expression\', \'type\': \'string\'}}, \'required\': [\'expression\'], \'title\': \'CalculatorParams\', \'type\': \'objec

In [12]:
agent.get_trace_history()

[{'id': 'evt_000001',
  'session_id': 'trace_3f71c3d757814feeae14ff3b2f12d30f',
  'turn_id': 'turn_0001',
  'seq': 1,
  'type': 'user_message',
  'timestamp': '2026-04-09T22:16:56.349187',
  'role': 'user',
  'content': '使用工具翻译下面的文字到英语并判断这个工具正确吗:\n你是谁，在哪里 \n 并帮我计算3^22',
  'metadata': {}},
 {'id': 'evt_000002',
  'session_id': 'trace_3f71c3d757814feeae14ff3b2f12d30f',
  'turn_id': 'turn_0001',
  'seq': 2,
  'type': 'tool_call',
  'timestamp': '2026-04-09T22:17:07.917236',
  'role': 'assistant',
  'content': '',
  'metadata': {'mode': 'tool', 'stream': True},
  'parent_id': 'evt_000001',
  'round': 1,
  'tool_name': 'translate_tool',
  'tool_args': {'text': '你是谁，在哪里'},
  'tool_call_id': 'call_c882f4a0fc8c467894513fbe6cbdd2a2'},
 {'id': 'evt_000003',
  'session_id': 'trace_3f71c3d757814feeae14ff3b2f12d30f',
  'turn_id': 'turn_0001',
  'seq': 3,
  'type': 'tool_result',
  'timestamp': '2026-04-09T22:17:07.918303',
  'role': 'tool',
  'content': 'Translated: 你是谁，在哪里',
  'metadata': {'mode':

In [8]:
agent.llm=EasyLLM(model="gpt-5.4",provider="openai_responses")

2026-04-09 22:18:28,841 | INFO | EasyLLM 初始化完成: provider=openai_responses, model=gpt-5.4


In [ ]:
agent.save_session("test_00001")

In [ ]:
await agent.astream_invoke("我们刚才聊了什么")

In [ ]:
manager=agent.skill_manager
prompt=manager.build_skills_prompt()
print(prompt)

In [ ]:
from skill.registry import SkillRegistry
skill_manage=SkillRegistry()
registered_names = skill_manage.discover_from_directory("./test_skills/")


In [ ]:
print(skill_manage.list_available())

In [ ]:
from skill.folder_loader import FolderSkillLoader
c_skill=FolderSkillLoader.load("./real_skills/crypto_skill/")

In [ ]:
print(c_skill.get_prompt())

In [ ]:
skill_manage.discover_from_directory("./real_skills/")

In [ ]:
print(skill_manage.list_available())


In [ ]:
crypto_skill=skill_manage.create('crypto_skill')
agent.with_skill(crypto_skill)
print(agent.get_enhanced_prompt())

In [ ]:
agent.invoke("i am a boy from china的 SHA-256 哈希值是什么")

In [ ]:
from memory.V2.WorkingMemory import WorkingMemory
from memory import MemoryConfig,MemoryManage
from memory.V2.Embedding.HuggingfaceEmbeddingModel import HuggingfaceEmbeddingModel
config = MemoryConfig(max_capacity=20)
working_memory = WorkingMemory(config)
mm = MemoryManage(
            config=config,
            user_id="test_integration_user",
            enable_working=True,
            working_memory=working_memory,
            enable_episodic=False,
            enable_semantic=False,
            enable_perceptual=False,
        ) 

In [ ]:
agent.with_memory(mm)
print(agent.get_enhanced_prompt())

In [ ]:
from skill.registry import SkillRegistry
from skill.builtin.calculator_skill import CalculatorSkill

# 1. 把所有 Skill 注册到全局 Registry（启动时一次性完成）
registry = SkillRegistry.instance()
registry.register_class(CalculatorSkill)
# 为搜索提供元信息
registry.update_metadata("calculator", description="数学计算工具", tags=["math", "compute"])
registry.discover_from_directory("./real_skills/")
# 也可以从目录批量发现
# registry.discover_from_directory("./skills/")

# 2. 创建 Agent（不预加载任何 Skill）
agent1 = BasicAgent(name="assistant", llm=llm, verbose_thinking=True)
agent1.with_skill(MetaSkill(registry,manager=agent1.skill_manager))
print(agent1.get_enhanced_prompt())

In [ ]:
agent1.invoke("i am a boy from china的 SHA-256 哈希值是什么")